# migrantBuddy - Experiment Findings

Organized by pipeline stage (04 retrieval, 05 generation, 06 eval). For each
stage: what was tested, every metric produced, and the conclusion drawn.
Model names and values are taken directly from the notebooks as they
currently stand -- update this file whenever a notebook produces a new
result worth keeping.

## Stage 04 -- Retrieval

### What we tested

- `04_retrieval.ipynb`: built the four retrieval functions (`dense`, `bm25`,
  `hybrid`, `hybrid_rerank` with `ms-marco-MiniLM-L-6-v2`) and did a
  qualitative-only side-by-side read of their top-3 results on 10 sample
  queries. No scored metrics computed here.
- `04b_retrieval_sealion_rerank.ipynb`: quantitative benchmark of 6 methods
  (`dense`, `bm25`, `hybrid`, and `hybrid` reranked by each of
  `ms-marco-MiniLM-L-6-v2`, `bge-reranker-v2-m3`, and
  `aisingapore/SEA-LION-E5-Embedding-600M`) against 10 labeled queries (7
  non-English) over the 16-chunk corpus, `top_k=5`, metrics computed at
  `k=3`.

### Metrics (`04b_retrieval_sealion_rerank.ipynb`)

| Method | MRR | hit@3 | precision@3 | recall@3 | nDCG@3 | Avg latency |
|---|---|---|---|---|---|---|
| `dense` | 0.875 | 0.900 | 0.300 | 0.900 | 0.863 | 137ms |
| `bm25` | 0.403 | 0.600 | 0.200 | 0.600 | 0.439 | 0.1ms |
| `hybrid` | 0.783 | 0.900 | 0.300 | 0.900 | 0.813 | 55ms |
| `hybrid_rerank` (ms-marco-MiniLM-L-6-v2) | 0.425 | 0.500 | 0.167 | 0.500 | 0.426 | 267ms |
| `hybrid_rerank` (bge-reranker-v2-m3) | 0.817 | 1.000 | 0.333 | 1.000 | 0.863 | 4772ms |
| **`hybrid_rerank` (SEA-LION-E5-Embedding-600M)** | **0.950** | **1.000** | **0.333** | **1.000** | **0.963** | 4872ms |

Per-query detail (reciprocal rank) for the SEA-LION reranker -- perfect on
every non-English query except one:

| Query language | RR |
|---|---|
| en (overtime) | 1.000 |
| en (salary timing) | 1.000 |
| ms | 1.000 |
| ta | 1.000 |
| my | 1.000 |
| th | 1.000 |
| vi | 1.000 |
| en (repatriation) | 1.000 |
| en (medical insurance) | 0.500 |
| en (contact MOM) | 1.000 |

### Conclusion

- `bm25` alone is weak on this corpus (MRR 0.403) -- non-English queries share
  almost no lexical tokens with the English-source text.
- `hybrid` (RRF fusion of dense + bm25) scores *lower* than `dense` alone
  (0.783 vs 0.875) -- fusing in a weak BM25 signal dilutes dense's already-good
  ranking at this corpus size.
- `ms-marco-MiniLM-L-6-v2` (English-only-trained) makes things *worse* than no
  reranking at all (0.425) -- fails outright (RR=0) on Malay/Tamil/Burmese/
  Vietnamese queries in the underlying per-query data.
- `bge-reranker-v2-m3` recovers most cross-lingual cases (0.817) but still
  drops rank on Malay and Vietnamese.
- **`SEA-LION-E5-Embedding-600M` is the chosen reranker** -- the only one to
  score a perfect RR=1.000 on every non-English query (medical-insurance is
  the one query all three rerankers miss alike, likely a genuinely ambiguous
  query rather than a language issue). Costs the same latency as
  bge-reranker-v2-m3 (~4.8-4.9s), so there's no speed reason to pick the
  weaker one.
- **Caveat:** SEA-LION-E5 is a bi-encoder (cosine similarity), not a true
  cross-encoder like the other two rerankers -- an accepted tradeoff given
  the accuracy gap.
- **Applied:** `05_generation.ipynb` and `06_eval.ipynb` use `hybrid` +
  SEA-LION reranking (`RERANKER_MODEL_NAME`) as the default retrieval method.

## Stage 05 -- Generation

### What we tested

- `05_generation.ipynb`: retrieval = `hybrid_rerank` (SEA-LION reranker,
  Stage 04's choice), generation model = `qwen3:8b`. Smoke test: 10 sample
  queries (7 non-English), read the generated answers for groundedness and
  honesty. No scored metric computed in this notebook.
- `05b_generation_sealion.ipynb`: identical retrieval, generation model
  swapped to `aisingapore/Llama-SEA-LION-v3-8B-IT` (an 8B instruction-tuned
  Llama-3.1-based model, continued-pretrained on SEA languages -- distinct
  from the SEA-LION-E5 reranker, a different model entirely despite the same
  family name). Same 10 sample queries, results saved to
  `data/processed/eval_results/05b_generation_aisingapore_llama_sea_lion_v3_8b_it.json`
  for later scoring. No scored metric computed here either.

### Metrics

None. Stage 05 only produces raw generated answers (qualitative smoke tests)
-- it does not run Ragas or any other scoring. Quantitative faithfulness /
answer_relevancy numbers for these same answers are Stage 06's job.

Both models produced plausible, on-topic, correctly-formatted answers across
all 10 sample queries in every tested language (en/ms/ta/my/th/vi) -- neither
produced an obviously broken or off-topic response on manual read. Stage 05
itself has no quantitative winner, since nothing here was scored -- see Stage
06 below for the Ragas comparison (`06b_eval_sealion_generation.ipynb` has now
been run).

## Stage 06 -- Eval

### What we tested

Runs A-C below are from `06_eval.ipynb`, scoring `qwen3:8b`-generated
answers with Ragas `Faithfulness` and `AnswerRelevancy`. Judge: `llama3.1:8b`
(separate from the generation model). Embeddings: `BAAI/bge-m3`.

- **Run A:** retrieval = `dense`, 10 labeled queries (the original set,
  before Stage 04's reranker swap was applied here).
- **Run B:** retrieval = `hybrid_rerank` (SEA-LION, Stage 04's choice), same
  10 labeled queries.
- **Run C:** retrieval = `hybrid_rerank` (SEA-LION), trimmed to 8 labeled
  queries (Burmese and Thai removed -- see conclusion below).
- **Run A (trimmed):** `06_eval.ipynb` Step 7 -- Run A rerun on the same
  8-query trimmed set as Run C, so dense and hybrid_rerank are now directly
  comparable query-for-query, not just at different query counts.

`06b_eval_sealion_generation.ipynb` runs the identical harness (same 8-query
set, same judge, same retrieval) against `aisingapore/Llama-SEA-LION-v3-8B-IT`
(Stage 05's SEA-LION generation experiment) instead of `qwen3:8b` -- isolating
the generation model as the only variable versus Run C.

### Metrics

| Run | Retrieval | Generation model | Query count | Avg faithfulness | Avg answer_relevancy |
|---|---|---|---|---|---|
| A | `dense` | `qwen3:8b` | 10 | 0.846 | 0.792 |
| B | `hybrid_rerank` (SEA-LION) | `qwen3:8b` | 10 | 0.771 | 0.774 |
| C | `hybrid_rerank` (SEA-LION) | `qwen3:8b` | 8 | 0.794 | 0.828 |
| A (trimmed) | `dense` | `qwen3:8b` | 8 | 0.794 | 0.826 |
| 06b | `hybrid_rerank` (SEA-LION) | `Llama-SEA-LION-v3-8B-IT` | 8 | **0.900** | 0.813 |

Notable per-query results feeding the conclusion below:

| Query | Run A faithfulness | Run B faithfulness |
|---|---|---|
| Burmese (`my`) | 1.0 | 0.25 |
| Thai (`th`) | 0.0 | 0.0 |

Run C's two flagged queries (faithfulness < 0.7): "When must my employer pay
my salary?" (0.6) and "How can I contact MOM?" (0.6) -- both added plausible
domain claims (tax withholding timelines, a "myMOM Portal") not clearly
present in the exact retrieved chunks, rather than clear fabrication.

Run A (trimmed)'s one flagged query is also "How can I contact MOM?", but far
more severe -- faithfulness=0.091, the lowest score in any run in this
notebook. The `dense`-retrieved answer added a "Get started" form name and a
15-minute time estimate the judge couldn't verify against the retrieved
chunks. Every other Run A (trimmed) query scored >=0.75.

06b's lowest-faithfulness query is "Who pays repatriation costs when my Work
Permit ends?" (0.75) -- no query drops below 0.7, unlike Run C's two flagged
queries above.

### Conclusion

- The faithfulness drop from Run A to Run B (0.846 -> 0.771) is **not** a
  real regression caused by the SEA-LION reranker. `(1.0 - 0.25) / 10 =
  0.075`, exactly the size of the overall drop -- the entire effect traces to
  the Burmese query alone, most likely `qwen3:8b` generation randomness (no
  `temperature`/`seed` pinned on the Ollama call), not a retrieval problem
  (SEA-LION ranked the correct chunk #1 for that exact query in Stage 04).
- Thai's faithfulness=0.0 is identical in both runs A and B -- a standing
  `llama3.1:8b` judge limitation verifying Thai-language claims against
  English-language context, unrelated to which retrieval method is used.
- **Decision:** Burmese and Thai were dropped from this notebook's
  `LABELED_QUERIES` (Run C, now 8 queries) so judge/generation noise stops
  dominating a 10-query average. Both languages remain in Stage 04's
  retrieval-only benchmark, where SEA-LION handled them correctly.
- **qwen3:8b baseline: 0.794 faithfulness / 0.828 answer_relevancy** on the
  clean 8-query set (Run C).
- **Dense vs hybrid_rerank barely matters on the 8-query set:** Run A
  (trimmed) scores **0.794 / 0.826** -- statistically indistinguishable from
  Run C's 0.794 / 0.828, despite using a different retrieval method
  (`dense` vs SEA-LION `hybrid_rerank`). This is a *generation*-eval result,
  not a retrieval-quality one -- it says qwen3:8b's answer faithfulness
  doesn't depend much on which of these two retrieval methods fed it context
  on this particular query set, not that the methods retrieve equally well
  (Stage 04's MRR benchmark already showed hybrid_rerank ahead there). Both
  runs are dragged down by the same query ("How can I contact MOM?"), though
  far more severely for dense (faithfulness=0.091 vs Run C's 0.6) -- a
  reminder that per-query outliers, not the retrieval method, are driving
  most of the variance at this sample size.
- **SEA-LION generation (06b) vs qwen3:8b baseline:** faithfulness **0.900 vs
  0.794** (+0.106) -- a real edge, and every 06b query scores >=0.7 (no
  flagged rows), whereas Run C had two and Run A (trimmed) had one severe
  outlier. answer_relevancy is a near-wash, **0.813 vs 0.828/0.826**
  (~-0.015), well within the swing a single query flip causes on an 8-query
  set (see caveats below). Net read: SEA-LION's generation model is more
  faithful to retrieved context on this corpus and query set, with no
  meaningful cost to relevance -- but an 8-query directional signal, not a
  settled result, given how much both the Burmese/Thai episode and the
  "contact MOM" outlier above showed single-query noise can move these
  averages.

## Current pipeline state (as of this writeup)

| Stage | Choice | Status |
|---|---|---|
| Embedding | `BAAI/bge-m3` | Committed default, unchanged throughout |
| Retrieval | `hybrid` (BM25 + dense, RRF) | Stage 04 |
| Reranker | `aisingapore/SEA-LION-E5-Embedding-600M` | **Settled** -- Stage 04 |
| Generation model | `aisingapore/Llama-SEA-LION-v3-8B-IT` | **Settled** -- Stage 06 (0.900 faithfulness / 0.813 answer_relevancy vs qwen3:8b's 0.794 / 0.828). `qwen3:8b` remains in `05_generation.ipynb`/`06_eval.ipynb` as the historical baseline, unchanged, same as `04_retrieval.ipynb` keeping its original reranker after 04b settled on SEA-LION there |
| Judge (eval only) | `llama3.1:8b` | Unchanged |

## Open caveats across all results

- **Small eval set (8-10 queries):** individual query flips can swing
  averages by 0.05-0.1+. Directional signal only, not statistically robust --
  a larger labeled set would tighten this.
- **Non-determinism:** `qwen3:8b`'s Ollama calls have no fixed
  `temperature`/`seed`, so repeated runs of the same query can legitimately
  produce different-quality answers.
- **Judge limitations:** the Thai faithfulness=0.0 pattern suggests
  `llama3.1:8b` struggles with non-English claim verification specifically --
  worth keeping in mind for any future non-English Ragas result that looks
  anomalously low.